### Classe Strategy pour modèle

In [ ]:
class IStrategyModel(IStrategy):

    DOWN = 0
    UP = 1

    def __init__(self, startTrading, indexColPrediction = None):

        print("StrategYModel file StrategiesModel")

        self.serie = pd.Series([])
        self.indexColPrediction = indexColPrediction
        
        self.arrayTrain =  np.array([])

        self.tendency = None

        self.startTrading = startTrading
        
        self.model = None
        self.initModel = False
        self.seriePrediction = pd.Series([])
        
    def filterIn(self):
        self.arrayTrain = np.array(self.serie)
        
        self.arrayTrain = np.log(self.arrayTrain) # [math.log(x*100) for x in self.arrayTrain]

    def filterOut(self, unityPrice):
        return np.exp(unityPrice)/100

    def fit(self):
        pass
        
    def predict(self):
        prediction = self.predictor.forecast(1)

        return self.filterOut(prediction)

    def getAction(self, game, unityPrice, date, unityData=None, debug=False):
    
        action = Game.STILL

        currentPrice = unityPrice

        if unityData is None:
            unityData = unityPrice
            currentData = unityPrice
        else:
            currentData = unityData[self.indexColPrediction]
        

        #if(self.indexColPrediction is None):
        #    currentPrice = unityPrice
        #else:
        #    currentPrice = unityPrice[self.indexColPrediction]

        #print("current Price is:", currentPrice)

        if(debug):
            print("Strategy model -  current data: ", currentPrice )
            
        if(date < self.startTrading):
            #self.serie.at[date] = unityPrice
            self.serie.at[date] = unityData 
            return (action, 0)
        else:

            #self.serie.at[date] = unityPrice
            self.serie.at[date] = unityData

            if(self.initModel == False):
                self.filterIn()
                self.fit()
                self.initModel = True
            elif(len(self.serie) % 1000 == 0):
                self.filterIn()
                self.fit()
            else:
                self.filterIn()

            pricePrediction = self.predict()
            self.seriePrediction.at[date+timedelta(1)] = pricePrediction

            if(game.clock+1 < len(game.prices)):
                print("--- prediction/real value/ diff: ", pricePrediction, " / ", game.prices[game.clock+1]," / ", abs(pricePrediction-game.prices[game.clock+1]))
                
            if(self.seriePrediction.size == 0):
                tendency = None
            elif(pricePrediction < currentData):
                tendency = IStrategyModel.DOWN 
            else:
                tendency = IStrategyModel.UP
            
            if (self.tendency is None) or (tendency == self.tendency):
                action = Game.STILL
            elif(tendency != self.tendency):
                if tendency == IStrategyModel.DOWN:
                    action = Game.SELL
                else:
                    action = Game.BUY

            self.tendency = tendency

        if debug:
            print("Strategy model - Game account cash:", game.account.cash)
            print("Strategy model - current price:", currentPrice)
            print("Strategy model - ", int(game.account.cash / currentPrice))
    
        if action == Game.BUY:
            return (action , int(game.account.cash / currentPrice))
        elif action == Game.SELL:
            return (action , game.account.security)
        else:
            return (action, 0)
            


## Implémentation de strategies

In [ ]:
class StrategySimpleExpSmoothing(IStrategyModel):

    def __init__(self, startTrading):
        super().__init__(startTrading, None)
        
    def fit(self):
        
        self.model = SimpleExpSmoothing(self.serieTrain)
        m = 12
        alpha = 1/(2*m)
        self.predictor = self.model.fit(smoothing_level=alpha,optimized=False,use_brute=True)
        
        
class StrategyExponentialSmoothing(IStrategyModel):

    def __init__(self, startTrading, trend='add'):
        ''' 
        @param trend : 'add' or 'mul', default is 'add'
        '''
        super().__init__(startTrading, None)
        self.trend = trend
        
    def fit(self):
     
        self.model = ExponentialSmoothing(self.serieTrain,trend=self.trend)
        self.predictor = self.model.fit()

class StrategyARIMA(IStrategyModel):

    def __init__(self, startTrading):
        super().__init__(startTrading, None)
        
    def fit(self):
        self.model = ARIMA(self.serieTrain, order=(1, 1, 0))
        self.predictor = self.model.fit()

class StrategyNull(IStrategyModel):

    def __init__(self, startTrading):
        super().__init__(startTrading, None)
        
    def predict(self):
        return self.serie[-1]

class StrategyARCH(IStrategyModel):

    def __init__(self, startTrading):
        super().__init__(startTrading, None)
    
    def predict(self):
        return 

    def fit(self):
        print("Train set: ", len(self.serieTrain))
        self.model = arch_model(self.serieTrain)
        self.predictor = self.model.fit()
        
    
            


In [ ]:
class StrategyLSTM(IStrategyModel):

    def __init__(self, startTrading, n_features=1, n_steps=31, indexColPrediction=None):
        super().__init__(startTrading, indexColPrediction)

        self.n_steps = n_steps
        self.n_features = n_features

        if(self.n_features > 1):
            columns = [x for x in range(0, self.n_features)]
            self.serie = pd.DataFrame(columns=columns)
            print(self.serie)

        self.model = tf.keras.Sequential()
        self.model.add(tf.keras.layers.LSTM(5, activation='relu', input_shape=(self.n_steps, self.n_features)))

        if self.indexColPrediction is None :
            self.model.add(tf.keras.layers.Dense(self.n_features))
        else:
            self.model.add(tf.keras.layers.Dense(1))

        self.model.compile(optimizer='adam', loss='mse')
        
    def filterIn(self):
        
        self.arrayTrain = np.array(self.serie).astype('float32')
        print("FilterIn: ", len(self.serie), " ", self.arrayTrain.shape)
       
    def predict(self):

        current = self.arrayTrain[-self.n_steps:]
        
        print("self.arrayTrain: ", self.arrayTrain.shape, " ", self.n_steps)
        
        current = current.reshape((1, self.n_steps, self.n_features))
        yhat = self.model.predict(current, verbose=0)
        print(current, type(current), type(self.arrayTrain), " prediction:", yhat, yhat.shape)

        if yhat.shape == (1,1):
            yhat = yhat[0][0]

        return yhat

    def fit(self):
        
        X, Y = self.split_sequence(self.arrayTrain, n_steps=self.n_steps)
        X = X.reshape((X.shape[0], X.shape[1], self.n_features))

        print("StrategyLSTM - fit() - Y shape : ", np.shape(Y))
        print("StrategyLSTM - fit() - X shape : ", np.shape(X))
        
        self.model.fit(X, Y, epochs=200, verbose=0)
        

    # split a univariate sequence into samples
    def split_sequence(self, sequence, n_steps):
        ''' 
        Split a sequence with [[n_steps, Y], [n_steps, Y], ...] in order to learn from n_steps values to predict Y

        @return : (np.array(X), np.array(y)), array with the 
        '''
        X, y = list(), list()
        for i in range(len(sequence)):
            # find the end of this pattern
            end_ix = i + n_steps
            # check if we are beyond the sequence
            if end_ix > len(sequence)-1:
                break
            # gather input and output parts of the pattern
            seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
            X.append(seq_x)

            if(self.indexColPrediction is None):
                y.append(seq_y)
            else:
                y.append(seq_y[self.indexColPrediction])

        return np.array(X), np.array(y)
    
    def getAction(self, game, unityPrice, date, unityData=None, debug=False):
    
        action = Game.STILL

        currentPrice = unityPrice

        if unityData is None:
            unityData = unityPrice
            currentData = unityPrice
        else:
            currentData = unityData[self.indexColPrediction]
        

        #if(self.indexColPrediction is None):
        #    currentPrice = unityPrice
        #else:
        #    currentPrice = unityPrice[self.indexColPrediction]

        #print("current Price is:", currentPrice)

        if(debug):
            print("Strategy model -  current data: ", currentPrice )
            
        if(date < self.startTrading):
            #self.serie.at[date] = unityPrice
            if(debug):
                print("date: ", date, " ", self.serie.index)
                print("serie:", self.serie.head())
                print("Unity Data:", unityData)
                print("self.serie:\n", self.serie)

            #self.serie.at[date] = unityData 
            #self.serie.loc[date, :] = np.array(unityData).astype('float32')
            self.serie.loc[date, :] = unityData
            

            return (action, 0)
        else:

            #self.serie.at[date] = unityPrice
            #self.serie.at[date] = unityData
            #self.serie.loc[date, :] = np.array(unityData).astype('float32')
            self.serie.loc[date, :] = unityData
            
            if(debug):
                print("self.serie:\n", self.serie)

            if(self.initModel == False):
                self.filterIn()
                self.fit()
                self.initModel = True
            elif(len(self.serie) % 1000 == 0):
                self.filterIn()
                self.fit()
            else:
                self.filterIn()

            pricePrediction = self.predict()
            self.seriePrediction.at[date+timedelta(1)] = pricePrediction

            if(game.clock+1 < len(game.prices)):
                print("--- prediction/real value/ diff: ", pricePrediction, " / ", game.prices[game.clock+1]," / ", abs(pricePrediction-game.prices[game.clock+1]))
                
            if(self.seriePrediction.size == 0):
                tendency = None
            elif(pricePrediction < 0):
                tendency = IStrategyModel.DOWN 
            else:
                tendency = IStrategyModel.UP
            
            if (self.tendency is None) or (tendency == self.tendency):
                action = Game.STILL
            elif(tendency != self.tendency):
                if tendency == IStrategyModel.DOWN:
                    action = Game.SELL
                else:
                    action = Game.BUY

            self.tendency = tendency

        if debug:
            print("Strategy model - Game account cash:", game.account.cash)
            print("Strategy model - current price:", currentPrice)
            print("Strategy model - ", int(game.account.cash / currentPrice))
    
        if action == Game.BUY:
            return (action , int(game.account.cash / currentPrice))
        elif action == Game.SELL:
            return (action , game.account.security)
        else:
            return (action, 0)
